In [7]:
import os
import json
import time
import requests
import pandas as pd
import hashlib
from datetime import datetime


# Configuration


collection_id = 189
RAW_PATH = "raw_data"

os.makedirs(RAW_PATH, exist_ok=True)


# Step 1 - Get Collection Metadata


metadata_url = f"https://api-production.data.gov.sg/v2/public/api/collections/{collection_id}/metadata"

response = requests.get(metadata_url)

if response.status_code != 200:
    raise Exception("Failed to retrieve collection metadata.")

metadata = response.json()

# print(json.dumps(metadata, indent=2)[:2000])


# Step 2 - Get Child Dataset IDs

child_datasets = metadata["data"]["collectionMetadata"]["childDatasets"]

print(f"\nFound {len(child_datasets)} datasets")

# for dataset in child_datasets:
#     print(dataset)


# Step 3 - Download Each Dataset


all_dataframes = []

for i, dataset in enumerate(child_datasets):

    file_path = os.path.join(RAW_PATH, f"hdb_file_{i+1}.csv")


    # Load file if exists

    if os.path.exists(file_path):

        # print(f"\nLoading cached file: {file_path}")

        df = pd.read_csv(file_path)

        all_dataframes.append(df)

        continue

    print("\n" + "=" * 50)
    # print(f"Downloading Dataset {i+1}")
    # print(dataset)

    poll_url = f"https://api-open.data.gov.sg/v1/public/api/datasets/{dataset}/poll-download"

    response = requests.get(poll_url)

    # Accept both 200 and 201
    if response.status_code not in [200, 201]:
        print("HTTP request failed.")
        continue

    poll_data = response.json()

    if poll_data["code"] != 0:
        print("API Error:")
        print(poll_data)
        continue

    csv_url = poll_data["data"]["url"]

    # print("Downloading CSV...")

    try:
        df = pd.read_csv(csv_url)

        df["dataset_id"] = dataset

        print(f"Rows downloaded: {len(df)}")

        # Save locally
        df.to_csv(file_path, index=False)

        print(f"Saved: {file_path}")

        all_dataframes.append(df)

    except Exception as e:
        print(f"Error reading CSV: {e}")

    time.sleep(1)


# Step 4 - Create Master DataFrame

if len(all_dataframes) == 0:
    raise Exception("No datasets were downloaded.")

df_master = pd.concat(all_dataframes, ignore_index=True)

# print("\n" + "=" * 50)
# print("MASTER DATAFRAME CREATED")
# print("=" * 50)

print("Shape:", df_master.shape)

print("\nColumns:")
# print(df_master.columns.tolist())

# print(df_master.head())

# Step 5 - Save Master Dataset


master_file = os.path.join(RAW_PATH, "master_hdb_dataset.csv")

df_master.to_csv(master_file, index=False)

# Step 6 - Data Profiling

# print("\nData Types")
# print(df_master.dtypes)

print("\nMissing Values")
print(df_master.isnull().sum())

print("\nSummary Statistics")
# print(df_master.describe(include="all"))
len(df_master)


Found 5 datasets
Shape: (982011, 12)

Columns:

Missing Values
month                       0
town                        0
flat_type                   0
block                       0
street_name                 0
storey_range                0
floor_area_sqm              0
flat_model                  0
lease_commence_date         0
remaining_lease        709050
resale_price                0
dataset_id                  0
dtype: int64

Summary Statistics


982011

In [8]:


# Create Output Folders


folders = ["cleaned", "transformed", "failed", "hashed"]

for folder in folders:
    os.makedirs(folder, exist_ok=True)


# Create Working Copy

df = df_master.copy()

print("Original Shape:", df.shape)


# Standardize Column Names


df.columns = (
    df.columns
      .str.strip()
      .str.lower()
      .str.replace(" ", "_")
)


# Convert Data Types

numeric_cols = [
    "resale_price",
    "floor_area_sqm",
    "lease_commence_date"
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

if "month" in df.columns:
    df["month"] = pd.to_datetime(
        df["month"],
        format="%Y-%m",
        errors="coerce"
    )


# Trim Text Columns

text_cols = [
    "town",
    "flat_type",
    "block",
    "street_name",
    "storey_range",
    "flat_model"
]

for col in text_cols:
    if col in df.columns:
        df[col] = (
            df[col]
            .astype(str)
            .str.strip()
            .str.upper()
        )

# Validation Rules


valid_rows = (
    (df["resale_price"] > 0) &
    (df["floor_area_sqm"] > 0) &
    (df["lease_commence_date"].between(1960, datetime.now().year)) &
    (df["month"].notna())
)

df_failed = df.loc[~valid_rows].copy()
df_clean = df.loc[valid_rows].copy()

print("Invalid Records:", len(df_failed))


# Duplicate Handling
# Business Key = All Columns Except resale_price
# Keep Higher Resale Price


business_key = [
    c for c in df_clean.columns
    if c != "resale_price"
]

# Sort by resale price descending
df_clean = df_clean.sort_values(
    by="resale_price",
    ascending=False
)

duplicate_mask = df_clean.duplicated(
    subset=business_key,
    keep="first"
)

duplicate_rows = df_clean[duplicate_mask].copy()

df_failed = pd.concat(
    [df_failed, duplicate_rows],
    ignore_index=True
)

df_clean = df_clean[~duplicate_mask].copy()

print("Duplicate Records Removed:", len(duplicate_rows))


# Remaining Lease

current_year = datetime.now().year

# Create a numeric remaining lease column
if "remaining_lease" not in df_clean.columns:
    df_clean["remaining_lease_years"] = (
        99 - (current_year - df_clean["lease_commence_date"])
    )
else:
    # Keep the original text column
    df_clean["remaining_lease_years"] = (
        99 - (current_year - df_clean["lease_commence_date"])
    )

# Ensure no negative values
df_clean["remaining_lease_years"] = (
    df_clean["remaining_lease_years"]
    .clip(lower=0)
)


# Save Cleaned Dataset


df_clean.to_csv(
    "cleaned/cleaned_dataset.csv",
    index=False
)

print("Cleaned Dataset Saved")


# Transformations


df_transformed = df_clean.copy()

df_transformed["price_per_sqm"] = (
    df_transformed["resale_price"] /
    df_transformed["floor_area_sqm"]
)

df_transformed["transaction_year"] = (
    df_transformed["month"].dt.year
)

df_transformed["transaction_month"] = (
    df_transformed["month"].dt.month
)


# Save Transformed Dataset


df_transformed.to_csv(
    "transformed/transformed_dataset.csv",
    index=False
)

print("Transformed Dataset Saved")


# Failed Dataset

df_failed.to_csv(
    "failed/failed_dataset.csv",
    index=False
)

print("Failed Dataset Saved")


# Hashing


df_hashed = df_clean.copy()

identifier_columns = [
    "month",
    "town",
    "block",
    "street_name",
    "flat_type"
]

def generate_hash(row):

    value = "|".join(
        str(row[col])
        for col in identifier_columns
    )

    return hashlib.sha256(
        value.encode("utf-8")
    ).hexdigest()

df_hashed["hashed_identifier"] = (
    df_hashed.apply(generate_hash, axis=1)
)

df_hashed.to_csv(
    "hashed/hashed_dataset.csv",
    index=False
)

print("Hashed Dataset Saved")

# Summary

print("\n ETL SUMMARY ")
print(f"Original Records      : {len(df_master)}")
print(f"Cleaned Records       : {len(df_clean)}")
print(f"Failed Records        : {len(df_failed)}")
print(f"Transformed Records   : {len(df_transformed)}")
print(f"Hashed Records        : {len(df_hashed)}")

print("\nOutput Files Created")
print("cleaned/cleaned_dataset.csv")
print("transformed/transformed_dataset.csv")
print("failed/failed_dataset.csv")
print("hashed/hashed_dataset.csv")

Original Shape: (982011, 12)
Invalid Records: 0
Duplicate Records Removed: 29347
Cleaned Dataset Saved
Transformed Dataset Saved
Failed Dataset Saved
Hashed Dataset Saved

 ETL SUMMARY 
Original Records      : 982011
Cleaned Records       : 952664
Failed Records        : 29347
Transformed Records   : 952664
Hashed Records        : 952664

Output Files Created
cleaned/cleaned_dataset.csv
transformed/transformed_dataset.csv
failed/failed_dataset.csv
hashed/hashed_dataset.csv
